# Hybrid Forecasting With BTC/ETH

## Paper Narrative

The main story here is the simplest multimodal forecasting question: **can we use Polymarket as a prior and a liquid crypto market as an external update signal**.

This benchmark answers the supporting paper question:
- does external crypto context improve terminal-outcome prediction beyond what is already encoded in the Polymarket state,
- is any gain concentrated at longer horizons,
- and is naive feature fusion enough, or do we need gating and trust-aware updates.

For the paper, this is a sanity-check benchmark for the multimodal story: if a naive hybrid already helps, that is strong evidence in favor of external covariates; if not, it still tells us what kind of modeling is missing.

## Everyday Intuition: Why This Might Work

The intuition is simple:
- Polymarket already provides a prior over the event;
- `BTC/ETH` may reflect part of the relevant public information faster;
- if the external market has already moved and the event market has not fully absorbed it yet, a hybrid model may produce a better terminal forecast.

## What Data We Will Use

- the main terminal forecasting dataset from Polymarket;
- snapshots `24h`, `72h`, and `168h` before resolution;
- aligned `btc_usd` and `eth_usd` 5-minute data;
- rolling returns and volatility features for crypto.

## What Metrics To Track

- `log_loss` as the main metric;
- `brier` and `roc_auc` as supporting metrics;
- horizon breakdowns showing where crypto helps or does not help.

## What Models To Train

- `market_price`;
- a `market-only` supervised baseline;
- a `crypto-only` baseline;
- a `hybrid` baseline as the first sanity check for the multimodal story.

## How To Read This Notebook

This is a research tutorial for the simplest multimodal extension of terminal forecasting.

Reading order:
1. We start from the main terminal-forecasting benchmark.
2. We add aligned `BTC/ETH` features at the snapshot time.
3. We compare `market_price`, `market-only`, `crypto-only`, and `hybrid`.
4. Then we inspect whether crypto helps more at distant horizons.

## Task

**What we test:**
- whether external crypto context improves final-outcome prediction beyond what is already contained in the Polymarket state.

**Input:**
- a market snapshot at one of the horizons before resolution;
- Polymarket history and metadata;
- aligned `BTC/ETH` returns and volatility features.

**Target:**
- the final binary market outcome.

**Why this task matters:**
- it is the most direct test of the idea “market prior + external update”;
- it shows whether an external liquid market adds genuinely new information or merely duplicates what is already encoded in the market price.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "benchmarks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "benchmarks"))

from benchmark_utils import (
    add_time_features,
    build_multi_horizon_terminal_dataset,
    connect,
    default_feature_columns,
    load_eligible_markets,
    load_probabilities_for_markets,
    rolling_time_splits,
)
from covariate_utils import asof_join_covariates, load_external_covariates

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

DOMAINS = ["geopolitics", "finance_economy"]
MAX_MARKETS_PER_DOMAIN = 200
HORIZONS_HOURS = (24, 72, 168)
MAX_SNAPSHOT_STALENESS_HOURS = 12
DB_PATH = REPO_ROOT / "db" / "resolved_probability_dataset.sqlite"
COVARIATE_PATH = REPO_ROOT / "data" / "external_covariates"


## Dataset Construction

Here we extend the main terminal-forecasting dataset with external covariates.

What happens:
- a snapshot dataset is built at horizons `24h`, `72h`, and `168h`;
- `BTC/ETH 5m` history is loaded;
- rolling returns / volatility features are computed;
- everything is then aligned by `cutoff_timestamp_utc`.

**Unit of evaluation:**
- one row = one market at one horizon before resolution + external crypto context at the same timestamp.

In [2]:
conn = connect(DB_PATH)
markets_df = pd.concat(
    [
        load_eligible_markets(conn, domain=domain, max_markets=MAX_MARKETS_PER_DOMAIN)
        for domain in DOMAINS
    ],
    ignore_index=True,
)
probabilities_df = load_probabilities_for_markets(conn, markets_df["market_id"].tolist())
terminal_df = build_multi_horizon_terminal_dataset(
    markets_df,
    probabilities_df,
    horizons_hours=HORIZONS_HOURS,
    max_snapshot_staleness_hours=MAX_SNAPSHOT_STALENESS_HOURS,
)
terminal_df = add_time_features(terminal_df)

summary = pd.DataFrame(
    {
        "domains": [", ".join(DOMAINS)],
        "markets": [terminal_df["market_id"].nunique()],
        "rows": [len(terminal_df)],
        "event_rate": [terminal_df["target"].mean()],
        "snapshot_start": [terminal_df["cutoff_timestamp_utc"].min()],
        "snapshot_end": [terminal_df["cutoff_timestamp_utc"].max()],
    }
)
display(summary)
terminal_df.head()


## Evaluation Protocol and Baselines

**Split:**
- rolling out-of-time splits by snapshot time.

**Metrics:**
- `log_loss` as the main probabilistic metric;
- `brier`;
- `roc_auc`.

**Baselines:**
- `market_price`;
- a `market-only` supervised baseline;
- a `crypto-only` baseline;
- a `hybrid` baseline.

It is especially important to inspect not only the aggregate result, but also the horizon breakdown: if crypto helps anywhere, it is most natural to expect gains at longer horizons.

In [3]:
covariates_df = load_external_covariates(COVARIATE_PATH)
covariates_df = covariates_df[covariates_df["series_id"].isin(["btc_usd", "eth_usd"])].copy()

price_wide = (
    covariates_df.pivot_table(
        index="timestamp_utc",
        columns="series_id",
        values="close",
        aggfunc="last",
    )
    .sort_index()
)
returns = price_wide.pct_change()

crypto_features = pd.DataFrame({"timestamp_utc": price_wide.index})
for series_id in ["btc_usd", "eth_usd"]:
    r = returns[series_id]
    crypto_features[f"{series_id}_ret_1h"] = r.rolling(12).sum().values
    crypto_features[f"{series_id}_ret_6h"] = r.rolling(72).sum().values
    crypto_features[f"{series_id}_ret_24h"] = r.rolling(288).sum().values
    crypto_features[f"{series_id}_vol_1h"] = r.rolling(12).std().values
    crypto_features[f"{series_id}_vol_24h"] = r.rolling(288).std().values

crypto_features["btc_eth_ret_gap_6h"] = (
    crypto_features["btc_usd_ret_6h"] - crypto_features["eth_usd_ret_6h"]
)

joined_df = asof_join_covariates(
    terminal_df,
    crypto_features,
    base_time_col="cutoff_timestamp_utc",
    max_age="10min",
)

crypto_cols = [col for col in joined_df.columns if col.startswith("btc_") or col.startswith("eth_")]
market_cols = default_feature_columns(joined_df, exclude=crypto_cols)
market_cols = [col for col in market_cols if col not in {"market_abs_error", "market_log_loss"}]

display(
    pd.DataFrame(
        {
            "joined_rows": [len(joined_df)],
            "joined_markets": [joined_df["market_id"].nunique()],
            "market_feature_count": [len(market_cols)],
            "crypto_feature_count": [len(crypto_cols)],
            "btc_missing_share": [joined_df["btc_usd_ret_1h"].isna().mean()],
        }
    )
)
joined_df.head()


## Results

In this block we test two questions:
- does crypto have any standalone signal (`crypto-only`);
- does adding crypto improve over the strong Polymarket baseline.

We then inspect the horizon breakdown separately to see whether useful signal may be concentrated only in early snapshots.

In [4]:
work_df = joined_df.dropna(subset=["target"]).copy()
splits = rolling_time_splits(work_df, time_col="cutoff_timestamp_utc", n_splits=4, min_train_fraction=0.5)

model_specs = {
    "market_price": None,
    "market_logistic": market_cols,
    "crypto_logistic": crypto_cols + ["horizon_hours", "hours_to_resolution"],
    "hybrid_logistic": market_cols + crypto_cols,
}

metric_rows = []
for model_name, feature_cols in model_specs.items():
    for train_df, test_df, meta in splits:
        y_test = test_df["target"].astype(int)
        if model_name == "market_price":
            p_test = np.clip(test_df["market_price_baseline"].astype(float).to_numpy(), 1e-6, 1.0 - 1e-6)
        else:
            pipeline = Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                    ("clf", LogisticRegression(max_iter=500, class_weight="balanced")),
                ]
            )
            y_train = train_df["target"].astype(int)
            pipeline.fit(train_df[feature_cols], y_train)
            p_test = pipeline.predict_proba(test_df[feature_cols])[:, 1]

        metric_rows.append(
            {
                "model": model_name,
                "fold": meta["fold"],
                "roc_auc": roc_auc_score(y_test, p_test),
                "brier": brier_score_loss(y_test, p_test),
                "log_loss": log_loss(y_test, np.clip(p_test, 1e-6, 1.0 - 1e-6)),
            }
        )

metrics_df = pd.DataFrame(metric_rows)
metrics_summary = (
    metrics_df.groupby("model")[["roc_auc", "brier", "log_loss"]]
    .agg(["mean", "std"])
    .round(4)
)
display(metrics_summary)

plot_df = metrics_df.melt(id_vars=["model", "fold"], var_name="metric", value_name="value")
g = sns.catplot(
    data=plot_df,
    x="model",
    y="value",
    col="metric",
    kind="bar",
    sharey=False,
    height=4,
    aspect=1.1,
)
g.set_xticklabels(rotation=25)
g.fig.suptitle("Hybrid Terminal Forecasting Baselines", y=1.05)
plt.show()


In [5]:
horizon_rows = []
for horizon_hours in sorted(work_df["horizon_hours"].unique()):
    horizon_df = work_df.loc[work_df["horizon_hours"] == horizon_hours].sort_values("cutoff_timestamp_utc").reset_index(drop=True)
    if len(horizon_df) < 30:
        continue

    split_idx = max(1, int(len(horizon_df) * 0.7))
    split_idx = min(split_idx, len(horizon_df) - 1)
    train_df = horizon_df.iloc[:split_idx].copy()
    test_df = horizon_df.iloc[split_idx:].copy()
    y_train = train_df["target"].astype(int)
    y_test = test_df["target"].astype(int)

    for model_name, feature_cols in {
        "market_price": None,
        "market_logistic": market_cols,
        "hybrid_logistic": market_cols + crypto_cols,
    }.items():
        if model_name == "market_price":
            p_test = np.clip(test_df["market_price_baseline"].astype(float).to_numpy(), 1e-6, 1.0 - 1e-6)
        else:
            pipeline = Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                    ("clf", LogisticRegression(max_iter=500, class_weight="balanced")),
                ]
            )
            pipeline.fit(train_df[feature_cols], y_train)
            p_test = pipeline.predict_proba(test_df[feature_cols])[:, 1]
        horizon_rows.append(
            {
                "horizon_hours": int(horizon_hours),
                "model": model_name,
                "log_loss": log_loss(y_test, np.clip(p_test, 1e-6, 1.0 - 1e-6)),
                "roc_auc": roc_auc_score(y_test, p_test),
            }
        )

horizon_df = pd.DataFrame(horizon_rows)
display(horizon_df.pivot(index="horizon_hours", columns="model", values="log_loss").round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=horizon_df, x="horizon_hours", y="log_loss", hue="model", marker="o", ax=axes[0])
axes[0].set_title("Log Loss by Horizon")
axes[0].set_xlabel("Horizon (hours before resolution)")
axes[0].set_ylabel("Log loss")

sns.lineplot(data=horizon_df, x="horizon_hours", y="roc_auc", hue="model", marker="o", ax=axes[1])
axes[1].set_title("ROC AUC by Horizon")
axes[1].set_xlabel("Horizon (hours before resolution)")
axes[1].set_ylabel("ROC AUC")
plt.tight_layout()


## Interpretation

This notebook is especially important for an honest scientific story.

If `hybrid` does not beat `market_price` on average, that is not a failure. It means:
- the external crypto market should not be plugged in as a naive always-on feature block;
- it should be used through a smarter mechanism: gating, trust-aware update, or regime-specific modeling.

If `hybrid` starts to help at longer horizons, that is exactly the kind of result that strongly supports the multimodal forecasting story for the paper.